# Homothetic Tube MPC — Interactive Lane-Change Playground

**Lateral vehicle dynamics · HT-MPC (proposed, homothetic tube) vs RT-MPC (fixed-tube robust MPC)**

Design your own lane-change manoeuvre (at constant speed, $v_x = 60$ km/h) and **watch both
controllers drive it, step by step**:
the vehicle, the tube cross-section it is currently guaranteed to stay in, and the
prediction the MPC is making at that instant — the nominal trajectory $z_{j|k}$ with the
tube $\alpha_{j|k}\Omega$ around it.

Both controllers share the *same* disturbance bound and the *same* RPI ellipsoid $\Omega$.
The only difference: HT-MPC scales its tube by $\alpha_k = \|e_k\|_{P^{-1}}$ at every step,
RT-MPC always uses the full worst-case tube ($\alpha_k \equiv 1$).

> **Reproducing the paper figures?** Use `LatVeh_Benchmark.ipynb` and `CSTR_Benchmark.ipynb`
> — those are the frozen reference implementations. This notebook imports `htmpc_latveh.py`,
> whose first section is a verbatim copy of the benchmark's model, uncertainty description,
> RPI computation, tightening and OCPs; only the reference generator, the prediction
> recording and the plots are new.

**How to use**

1. Run all cells (`Runtime ▸ Run all` in Colab, `Kernel ▸ Restart & Run All` in Jupyter).
2. In **Interactive playground**, pick a manoeuvre, move the sliders, press **Run**
   (~1 s), then press **▶ Play** on the figure or drag the time slider.

**What to look for**

* Top panel (HT-MPC): the thick bar at the vehicle is the *current* tube cross-section
  $\alpha_k\Omega$ — usually tiny. The shaded band ahead is the tube the OCP is planning with.
* Middle panel (RT-MPC): the current cross-section is always the full $\Omega$, and every
  applied input is tightened by the full worst case — that is why it steers much less
  aggressively (compare the RMS tracking error in the summary).
* Make the manoeuvre aggressive (slalom, short `t_hold`) and RT-MPC loses feasibility
  while HT-MPC keeps going; its panel freezes and the title says at which step.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Installs missing packages (needed on Colab) and fetches the engine module if this
# notebook is not sitting next to htmpc_latveh.py (e.g. opened directly in Colab).
import importlib.util, subprocess, sys, os, urllib.request

RAW_URL = "https://raw.githubusercontent.com/MiladBnit/homothetic-tube-mpc/main/htmpc_latveh.py"   # <- fill in

_missing = [p for p in ("cvxpy", "clarabel", "osqp", "scs", "plotly", "ipywidgets", "nbformat")
            if importlib.util.find_spec(p) is None]
if _missing:
    print("Installing:", " ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

if not os.path.exists("htmpc_latveh.py"):
    print("Downloading htmpc_latveh.py ...")
    urllib.request.urlretrieve(RAW_URL, "htmpc_latveh.py")

import numpy as np
import htmpc_latveh as H
print("Engine OK")

## 1 · Offline design (identical numbers to the paper)

Vehicle at $v_{x0} = 60$ km/h, $T_s = 0.1$ s, ancillary LQR gain $K$, shared 99 %
confidence bound, RPI ellipsoid $\Omega = \{e : e^\top P^{\star -1} e \le 1\}$, constraint
tightening and both OCPs with horizon $N = 25$. Runs once (~2 s).

In [ ]:
d = H.design(verbose=True)

## 2 · Interactive playground

| Control | Meaning |
|---|---|
| **Manoeuvre** | *Single lane change (paper)*: right → left lane at $t = 1.2$ s. *Slalom*: alternating lane changes every 1.5 s. *Custom waypoints*: a stair editor appears — draw the reference by setting the lane position for each of 8 time segments with the vertical sliders |
| **T_sim** | simulation length [s] (set automatically when a preset is chosen) |
| **confidence p** | shared confidence level of the Gaussian uncertainty bounds (paper: 0.99). Higher $p$ → larger $\Omega$, more tightening, less steering margin — at $p = 0.999$ even the paper manoeuvre becomes infeasible |
| **also animate α_k and δ_s** | adds two synchronised panels: the tube scale $\alpha_k$ and the steering angle $\delta_s$ with its bounds |

Everything else (vehicle, $\Delta v_x$ and noise realisations, horizon $N = 25$) is exactly as in
the paper. Press **Run**; the animation starts by itself, and **▶ Play** / the time slider
let you replay or scrub.

In [ ]:
import ipywidgets as W
from IPython.display import display, HTML

P_LEVELS = [0.80, 0.90, 0.95, 0.99, 0.995, 0.999]      # shared confidence level p
MENU     = ["Single lane change (paper)", "Slalom", "Custom waypoints"]
N_SEG    = 8                                           # segments of the custom stair reference
designs  = {0.99: d}                                   # offline designs, cached per p

# ── top row: manoeuvre | T_sim | confidence p ────────────────────────────────
w_mode = W.Dropdown(options=MENU, value=MENU[0], description="Manoeuvre",
                    style={"description_width": "75px"}, layout=W.Layout(width="330px"))
w_tsim = W.FloatSlider(5.0, min=2.0, max=15.0, step=0.5, description="T_sim [s]",
                       style={"description_width": "65px"}, layout=W.Layout(width="300px"),
                       readout_format=".1f")
w_p    = W.SelectionSlider(options=[(f"{p:g}", p) for p in P_LEVELS], value=0.99,
                           description="confidence p", style={"description_width": "90px"},
                           layout=W.Layout(width="300px"))

# ── custom tray: draw the staircase with vertical sliders (custom mode only) ─
_default_stair = [-1.75, -1.75, 1.75, 1.75, 0.0, 0.0, -1.75, -1.75]
seg = [W.FloatSlider(v, min=-2.25, max=2.25, step=0.25, orientation="vertical",
                     readout_format=".2f", layout=W.Layout(height="170px", width="52px"))
       for v in _default_stair]
seg_times = W.HTML()
tray = W.VBox([W.HTML("<b>Custom reference</b> &mdash; set the lane position e<sub>y</sub> [m] "
                      "for each time segment (each slider spans T_sim / 8):"),
               W.HBox(seg), seg_times])
tray.layout.display = "none"


def _refresh_seg_times(*_):
    dt = w_tsim.value / N_SEG
    seg_times.value = "".join(
        f"<span style='display:inline-block;width:52px;text-align:center;font-size:11px'>"
        f"{i*dt:.1f}&nbsp;s</span>" for i in range(N_SEG))


def _custom_string():
    dt = w_tsim.value / N_SEG
    return "; ".join(f"{i*dt:.3f}:{s.value}" for i, s in enumerate(seg))


def _on_mode(change):
    w_tsim.value = H.PRESETS[change["new"]]["T_sim"]
    tray.layout.display = "" if change["new"] == "Custom waypoints" else "none"


w_mode.observe(_on_mode, names="value")
w_tsim.observe(_refresh_seg_times, names="value")
_refresh_seg_times()

# ── run row ──────────────────────────────────────────────────────────────────
w_pan = W.Checkbox(False, description="also animate α_k and δ_s", indent=False)
w_run = W.Button(description="Run", button_style="success", icon="play",
                 layout=W.Layout(width="120px"))
out   = W.Output()


def _on_run(_):
    with out:
        out.clear_output(wait=True)
        p = w_p.value
        if p not in designs:                              # ~0.2 s: new Omega, tightening, OCPs
            designs[p] = H.design(p_conf=p, verbose=False)
        dd = designs[p]
        try:
            tr_p, tr_r, y_ref, dv = H.run(dd, w_mode.value, T_sim=w_tsim.value,
                                          custom=_custom_string())
        except ValueError as e:
            print(f"Invalid manoeuvre: {e}")
            return
        ttl = (f"{w_mode.value},  p = {p:g}  "
               f"(Ω half-width {dd.s_ey:.2f} m, steering margin {dd.u_bounds[0]-dd.tight.s_u[0]:.3f} rad)")
        fig = H.animate_overlay(dd, tr_p, tr_r, y_ref, title=ttl,
                                frame_ms=int(round(1000*dd.Ts)), panels=w_pan.value)
        # Plain HTML output -> exactly one figure in VS Code, JupyterLab and Colab.
        # Start the animation ourselves with the same timing as the Play button
        # (Plotly's built-in auto_play uses slow default timings).
        autoplay = ("Plotly.animate('{plot_id}', null, {frame: {duration: %d, redraw: false}, "
                    "transition: {duration: 0}, fromcurrent: true});" % int(round(1000*dd.Ts)))
        display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn",
                                 auto_play=False, post_script=autoplay)))


w_run.on_click(_on_run)
display(W.VBox([W.HBox([w_mode, w_tsim, w_p]), tray, W.HBox([w_run, w_pan]), out]))
_on_run(None)   # paper case on first run